# Label study: merged crowns against HOT's labels

Run on a GPU runtime. Set `REPO_SHA` and `MODEL` in the next cell. Two uses:

- `MODEL = "release"`: the release weights `NEON.pt`. Run setup, HOT data, predict (val) and the window check, then the label audit if `RUN_AUDIT`, and the summary. Run every cell except the test cell, which refuses this model because HOT test was already scored zero-shot; the sample views skip themselves.
- `MODEL = "<run id>"` (a fine-tuned run, e.g. `hot-platform`, with `best.pt` in `fair_trees/runs/<run id>`): run every cell in order. Test is scored once, at the window the check picks.

The model predicts crown boxes on every HOT validation unit at each window in `WINDOWS_M` (the window is the side of the square the model sees). The merge step turns touching crowns into `tree_group` boxes and keeps the rest as `tree` boxes. For each window, the merge settings (score threshold, gap, minimum crowns N) are tuned on HOT val. The window check starts from the smallest window and compares each larger one, in ascending order, with the current pick on the same val units (paired bootstrap). A larger window replaces the pick only if the 95% interval of its gain in mean F1 is above zero. The chosen setting is then scored once on HOT test. The audit profiles how each training source labels dense canopy.

The predictions are cached per unit on Drive (`fair_trees/label_study/cache-<MODEL>-<window>m`): re-running the predict cell after a disconnect resumes. Results are written to `experiments/results/label-study-<MODEL>` in the clone, with a copy of each file on Drive (`fair_trees/label_study/results-<MODEL>`); download the folder at the end to commit it.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
REPO_SHA = ""
MODEL = "release"  # "release" (NEON.pt), or a fine-tuned run id whose best.pt is on Drive
WINDOWS_M = (40.0, 60.0, 80.0)  # prediction windows compared on val
RUN_AUDIT = True
SEED = 0
MAX_REGIONS = 300  # regions per source in the label audit, sampled with SEED
# audit sources at pinned commits: restor/tcd main, and the CanopyRS gpkg branches (region-level files)
TCD_REVISION = "d97d4da0ebbb6e249ae95ac5e19656babd972eb2"
QUEBEC_REVISION = "16f3574ff8cfe798cf4f0c93b2f70d12468225f0"
SELVABOX_REVISION = "1f78a7a7fdcb27257df1f88efbf0b385c90c3dc9"

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

repo = Path("/content/fair_trees")
if MODEL == "release":
    WEIGHTS = DRIVE / "weights" / "NEON.pt"
else:
    WEIGHTS = DRIVE / "runs" / MODEL / "best.pt"
assert WEIGHTS.exists(), f"no weights at {WEIGHTS}"

# the first run (release weights, 40 m windows) cached its records as "cache"
legacy = DRIVE / "label_study" / "cache"
if legacy.exists() and not (DRIVE / "label_study" / "cache-release-40m").exists():
    legacy.rename(DRIVE / "label_study" / "cache-release-40m")


def cache_dir(w):
    """One record per unit on Drive, per model and window, so a disconnect loses nothing."""
    return DRIVE / "label_study" / f"cache-{MODEL}-{w:g}m"


RESULTS = repo / "experiments/results" / f"label-study-{MODEL}"
RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = (
    DRIVE / "label_study" / f"results-{MODEL}"
)  # a copy of every results file survives a disconnect
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)


def to_drive(path):
    """Copy a results file to Drive as well."""
    (DRIVE_RESULTS / path.name).write_bytes(path.read_bytes())

## HOT data

Labels and tile footprints for every split, imagery for validation and test. The masking cost is what masking `tree_group` polygons out of training would remove from HOT train.

In [ ]:
import json

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, masking_cost, read_hot

SPLITS = ("validation", "test")  # as named in tiles.parquet


def fetch(path):
    return hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION)


data = read_hot(fetch("stac/project-50661/labels.parquet"), fetch("stac/project-50661/tiles.parquet"))
images = {}  # tile_id -> encoded image
for split in SPLITS:
    table = pq.read_table(fetch(f"data/{split}-00000-of-00001.parquet"), columns=["tile_id", "image"])
    encoded = (v["bytes"] for v in table["image"].to_pylist())
    images.update(zip(table["tile_id"].to_pylist(), encoded, strict=True))
print(f"{len(data.trees):,} trees, {len(data.groups):,} groups, {len(images):,} val and test tiles")

cost = masking_cost(data, "train")
(RESULTS / "masking_cost.json").write_text(json.dumps(cost, indent=2))
to_drive(RESULTS / "masking_cost.json")
print(json.dumps(cost, indent=2))

## Predict (val)

One mosaic per unit (a connected run of tiles in a 16 × 16 tile block), at each window in `WINDOWS_M`; other settings are the serving defaults. Re-running the cell resumes: units with a record in that window's cache are skipped. Test units are predicted later, at the chosen window only.

In [ ]:
import shutil

import torch

from experiments.label_study.hot import split_units, unit_mosaic, unit_truth
from experiments.label_study.study import run_units
from fair_trees.params import InferenceParams
from fair_trees.serve import predict_mosaic
from fair_trees.train import build_model, torch_run_fn

SCRATCH = Path("/content/label_study_units")  # PNG chips of the unit being built
run_fn = torch_run_fn(build_model(WEIGHTS, trainable_layers=0), torch.device("cuda"))


def predictor(w):
    def predict(mosaic):
        return predict_mosaic(run_fn, mosaic, InferenceParams(window_m=w))

    return predict


def unit_builder(tile_ids, workdir):
    def build():
        shutil.rmtree(workdir, ignore_errors=True)  # left over from an interrupted run
        mosaic = unit_mosaic(images, tile_ids, workdir)
        return mosaic, unit_truth(data, mosaic, tile_ids)

    return build


def split_units_of(split):
    return [
        (unit_id, split, unit_builder(tile_ids, SCRATCH / unit_id))
        for unit_id, tile_ids in split_units(data.tiles[data.tiles["split"] == split], split)
    ]


val_units = split_units_of("validation")
builders = {unit_id: build for unit_id, _, build in val_units}
for w in WINDOWS_M:
    predicted = run_units(predictor(w), val_units, cache_dir(w))
    done = sum((cache_dir(w) / f"{unit_id}.npz").exists() for unit_id in builders)
    print(f"{w:g} m: predicted {predicted} units now; {done} / {len(val_units)} val units done")

## Window check

Every (score, gap, N) setting on HOT val, for each window; each window's best setting has the highest mean of tree F1 and group F1 (IoU 0.4). A HOT group part that touches the unit edge and covers under 50 m² is an ignore region in the group score: no cluster of crowns can match such a clipped sliver. Starting from the smallest window, each larger window's best setting is compared with the current pick's on the same val units (paired bootstrap), and replaces it only when the 95% interval of its gain in mean F1 is above zero (so 80 m is judged against 60 m if 60 m won, otherwise against 40 m). The plot shows the trade-off in N at the chosen window's score and gap.

In [ ]:
import matplotlib.pyplot as plt

from experiments.label_study.study import choose, load_records, pick_window, sweep


def cached(split, w):
    """The split's records cached at window w; all its units must be there, or the scores leave some out."""
    records = load_records(cache_dir(w), split)
    expected = len(split_units(data.tiles[data.tiles["split"] == split], split))
    assert len(records) == expected, (
        f"{len(records)} of {expected} {split} units cached at {w:g} m: rerun the predict cell"
    )
    return records


tables, settings, windows = {}, {}, {}
for w in WINDOWS_M:
    tables[w] = sweep(cached("validation", w))
    tables[w].to_csv(RESULTS / f"sweep_val_{w:g}m.csv", index=False)
    to_drive(RESULTS / f"sweep_val_{w:g}m.csv")
    settings[w] = choose(tables[w])
    row = tables[w][
        (tables[w]["score"] == settings[w]["score"])
        & (tables[w]["gap_m"] == settings[w]["gap_m"])
        & (tables[w]["min_crowns"] == settings[w]["min_crowns"])
    ].iloc[0]
    windows[f"{w:g}"] = {
        "setting": settings[w],
        "tree_f1": float(row["tree_f1"]),
        "group_f1": float(row["group_f1"]),
        "mean_f1": float(row["mean_f1"]),
    }
    print(f"{w:g} m: chosen {settings[w]}, mean F1 {row['mean_f1']:.3f}")

check = pick_window({w: cached("validation", w) for w in WINDOWS_M}, settings)
WINDOW_M = check["chosen_m"]
setting = settings[WINDOW_M]
window_check = {"windows": windows, "comparisons": check["comparisons"], "chosen_m": WINDOW_M}
(RESULTS / "window_check.json").write_text(json.dumps(window_check, indent=2))
to_drive(RESULTS / "window_check.json")
print(json.dumps(check["comparisons"], indent=2))
print(f"chosen window: {WINDOW_M:g} m, setting: {setting}")

table = tables[WINDOW_M]
print(table.sort_values("mean_f1", ascending=False, kind="stable").head(10).to_string(index=False))
curve = table[(table["score"] == setting["score"]) & (table["gap_m"] == setting["gap_m"])]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curve["min_crowns"], curve["tree_f1"], marker="o", label="tree F1")
ax.plot(curve["min_crowns"], curve["group_f1"], marker="o", label="group F1 (IoU 0.4)")
ax.axvline(setting["min_crowns"], color="grey", linestyle=":")
ax.set(xlabel="N (min crowns per group)", ylabel="F1 on HOT val")
ax.set_title(f"{WINDOW_M:g} m window, score > {setting['score']}, gap {setting['gap_m']} m")
ax.legend()
fig.savefig(RESULTS / "sweep_val.png", dpi=120, bbox_inches="tight")
to_drive(RESULTS / "sweep_val.png")
plt.show()

## Test, once

Fine-tuned runs only: the first line refuses the release weights, whose test score is already taken. Run this cell once, after the window check; do not tune on its output. It predicts the test units at the chosen window only, then scores the chosen setting. `groups_40_all` is the IoU 0.4 group score without the edge-sliver rule. The dense rows count only the tiles at least 30% inside HOT groups. After a disconnect, reload the result instead of scoring test again: `test = json.loads((DRIVE_RESULTS / "test.json").read_text())`.

In [ ]:
from experiments.label_study.study import score_test

assert MODEL != "release", "the release weights were already scored on test; use a fine-tuned run"
test_units = split_units_of("test")
builders |= {unit_id: build for unit_id, _, build in test_units}
predicted = run_units(predictor(WINDOW_M), test_units, cache_dir(WINDOW_M))
print(f"predicted {predicted} test units now at {WINDOW_M:g} m")

test = score_test(cached("test", WINDOW_M), setting)
(RESULTS / "test.json").write_text(json.dumps(test, indent=2))
to_drive(RESULTS / "test.json")


def fmt(value):
    return "n/a" if value is None else f"{value:.3f}"


def f1_with_ci(m):
    if m["f1"] is None:
        return "n/a"
    lo, hi = m["f1_ci"]
    return f"{m['f1']:.3f} [{lo:.3f}, {hi:.3f}]"


subsets = {"all": ("trees", "groups_40", "groups_25", "groups_40_all"), "dense": ("trees", "groups_40")}
for subset, keys in subsets.items():
    tiles = f", {test[subset]['tiles']} dense tiles" if subset == "dense" else ""
    print(f"{subset} ({test[subset]['units']} units{tiles})")
    for key in keys:
        m = test[subset][key]
        print(f"  {key:13} P {fmt(m['precision'])}  R {fmt(m['recall'])}  F1 {f1_with_ci(m)}")
print("predicted tree box side / HOT crown diameter:", test["diameter_ratio"])

## Sample views

Four test units at the chosen window (the two with the most dense tiles, two with none): predicted `tree` boxes (yellow) and `tree_group` boxes (magenta), HOT tree points (red) and HOT group boxes (cyan, dashed). Then one unit's unmatched tree boxes, to judge by eye whether they are trees HOT did not map. Fine-tuned runs only: it needs the test records.

In [ ]:
import random

import numpy as np
from matplotlib.patches import Rectangle

from experiments.label_study.merge import merge_crowns
from experiments.label_study.score import match_points
from experiments.label_study.study import DENSE_COVER
from fair_trees.decode import Detections

if MODEL == "release":
    print("no test records for the release weights; sample views skipped")
else:
    test_records = cached("test", WINDOW_M)
    dense_tiles = {r.unit_id: int((r.tile_cover >= DENSE_COVER).sum()) for r in test_records}
    most_dense = sorted(test_records, key=lambda r: dense_tiles[r.unit_id], reverse=True)[:2]
    none_dense = [r for r in test_records if dense_tiles[r.unit_id] == 0]
    views = most_dense + random.Random(SEED).sample(none_dense, 2)

    def merged_at_setting(rec):
        """The chosen setting's merge, and which of its tree boxes have their centre in a HOT group."""
        keep = rec.scores > setting["score"]
        dets = Detections(rec.boxes[keep], rec.scores[keep])
        merged = merge_crowns(dets, setting["gap_m"] / rec.gsd_m, setting["min_crowns"])
        return merged, rec.in_group[keep][merged.tree_index]

    def draw_boxes(ax, boxes, color, **style):
        style = {"linewidth": 1} | style
        for x0, y0, x1, y1 in boxes:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, **style))

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for ax, rec in zip(axes.flat, views, strict=True):
        merged, _ = merged_at_setting(rec)
        ax.imshow(builders[rec.unit_id]()[0].rgb)
        draw_boxes(ax, merged.trees.boxes, "yellow")
        draw_boxes(ax, merged.groups.boxes, "magenta", linewidth=2)
        draw_boxes(ax, rec.group_boxes, "cyan", linestyle="--")
        ax.scatter(*rec.tree_points.T, s=8, c="red")
        ax.set_title(f"{rec.unit_id} ({dense_tiles[rec.unit_id]} dense tiles)")
        ax.axis("off")
    fig.savefig(RESULTS / "sample_views.png", dpi=100, bbox_inches="tight")
    to_drive(RESULTS / "sample_views.png")
    plt.show()

    rec = views[2]  # no dense tile: most of its HOT trees are points
    merged, in_group = merged_at_setting(rec)
    trees = merged.trees.boxes[~in_group]  # the boxes score_test counts
    matched = [i for i, _ in match_points(trees, rec.tree_points)]
    unmatched = trees[np.isin(np.arange(len(trees)), matched, invert=True)]
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(builders[rec.unit_id]()[0].rgb)
    draw_boxes(ax, unmatched, "yellow", linewidth=2)
    ax.scatter(*rec.tree_points.T, s=8, c="red")
    ax.set_title(f"{rec.unit_id}: {len(unmatched)} of {len(trees)} tree boxes hold no HOT point")
    ax.axis("off")
    fig.savefig(RESULTS / "unmatched_trees.png", dpi=100, bbox_inches="tight")
    to_drive(RESULTS / "unmatched_trees.png")
    plt.show()

## Label audit

How each source labels dense canopy, clustered with the chosen window's gap and N. Set `RUN_AUDIT = False` to skip it. At most `MAX_REGIONS` regions per source:
- HOT: one region per split; trees are squares of their `diameter_crown`.
- OAM-TCD (`restor/tcd`, train and test): one region per 2048 px tile; category 2 is tree, 1 is canopy (group). Its 10 cm GSD is in EPSG:3395 units, so the ground GSD is 0.1 m × cos(latitude).
- NEON: the annotated training rasters of the raw archive on Drive, 10 cm.
- QuebecTrees and SelvaBox: the `gpkg` branch, one region per AOI file (a raster's train, valid or test area); its tiles on `main` overlap.

In [ ]:
# the cell imports all it uses rather than lean on earlier cells, hence the noqa on repeated imports
import json
import math
import random
import shutil  # noqa: F811
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download  # noqa: F811

from experiments.converters.neon import find_training_rasters
from experiments.label_study.audit import audit_table, coco_labels, geo_labels, neon_labels
from experiments.label_study.hot import MAX_DIAMETER_M, hot_labels


def pick(items):
    """At most MAX_REGIONS of the items, sampled with SEED."""
    items = sorted(items)
    return random.Random(SEED).sample(items, min(MAX_REGIONS, len(items)))


def concat(parts):
    """One (labels, regions) pair from a list of them."""
    return tuple(pd.concat(frames, ignore_index=True) for frames in zip(*parts, strict=True))


def tcd_labels():
    fs = HfFileSystem()
    files = sorted(fs.glob(f"datasets/restor/tcd@{TCD_REVISION}/data/*.parquet"))
    meta = pd.concat(pq.read_table(f, columns=["image_id", "crs"], filesystem=fs).to_pandas() for f in files)
    assert (meta["crs"] == "EPSG:3395").all(), "the GSD below assumes EPSG:3395 tiles"
    picked = pick(meta["image_id"].tolist())
    columns = ["image_id", "width", "height", "lon", "coco_annotations"]
    parts = []
    for f in files:
        table = pq.read_table(f, columns=columns, filesystem=fs, filters=[("image_id", "in", picked)])
        for row in table.to_pylist():
            image = {"id": row["image_id"], "width": row["width"], "height": row["height"]}
            coco = {"images": [image], "annotations": json.loads(row["coco_annotations"])}
            # 0.1 m is in EPSG:3395 units; the parquet's lat and lon columns are swapped: lon is the latitude
            gsd_m = 0.1 * math.cos(math.radians(row["lon"]))
            parts.append(coco_labels(coco, source="oam-tcd", gsd_m=gsd_m, tree_ids={2}, group_ids={1}))
    return concat(parts)


def neon_training_labels():
    raw_drive = DRIVE / "raw" / "neon"
    listed = {f["name"] for f in json.loads((raw_drive / "manifest.json").read_text())["files"]}
    raw_root = Path("/content/raw/neon")
    shutil.rmtree(raw_root, ignore_errors=True)
    # annotations.zip carries its own top folder; training.zip has RGB/, CHM/, ... at its root
    extract = (("annotations.zip", "annotations/", raw_root), ("training.zip", "RGB/", raw_root / "training"))
    for name, prefix, target in extract:
        assert name in listed, f"{name} is not in the raw manifest on Drive"
        with zipfile.ZipFile(raw_drive / name) as z:
            z.extractall(target, members=[m for m in z.namelist() if m.startswith(prefix)])
    return neon_labels(pick(find_training_rasters(raw_root)))


def gpkg_labels(repo_id, revision, source):
    """A CanopyRS gpkg-branch source, one region per AOI file; boxes belong to the AOI under their centre."""

    def get(name):
        return hf_hub_download(repo_id, name, repo_type="dataset", revision=revision)

    files = HfApi().list_repo_files(repo_id, repo_type="dataset", revision=revision)
    aois = pick(f for f in files if "_aoi_" in f)
    parts = []
    for stem in sorted({f.split("_aoi_")[0] for f in aois}):  # <raster>_labels_<masks|boxes>
        crowns = gpd.read_file(get(f"{stem}.gpkg"))
        utm = crowns.estimate_utm_crs()
        crowns = crowns.to_crs(utm)
        mine = [f for f in aois if f.startswith(f"{stem}_aoi_")]
        regions = pd.concat(gpd.read_file(get(f)).assign(region=f.removesuffix(".gpkg")) for f in mine)
        regions = regions[["region", "geometry"]].dissolve("region").reset_index().to_crs(utm)
        centres = gpd.GeoDataFrame(geometry=crowns.representative_point(), crs=utm)
        under = centres.sjoin(regions, predicate="within")
        under = under[~under.index.duplicated()]  # a centre on a shared AOI edge: first
        crowns = crowns.loc[under.index].assign(region=under["region"])
        parts.append(geo_labels(crowns, regions, source=source))
    return concat(parts)


if RUN_AUDIT:
    parts = {
        "hot": hot_labels(data),
        "oam-tcd": tcd_labels(),
        "neon": neon_training_labels(),
        "quebectrees": gpkg_labels("CanopyRS/QuebecTrees", QUEBEC_REVISION, "quebectrees"),
        "selvabox": gpkg_labels("CanopyRS/SelvaBox", SELVABOX_REVISION, "selvabox"),
    }
    audit = audit_table(parts, gap_m=setting["gap_m"], min_crowns=setting["min_crowns"])
    audit.to_csv(RESULTS / "audit.csv")
    to_drive(RESULTS / "audit.csv")
    print(audit.T.to_string(float_format=lambda v: f"{v:.3g}"))
    hot_trees = parts["hot"][0].query("cls == 'tree'")
    print(
        "HOT: crowns_per_ha counts all trees; crown sizes and the touching and HOT-rule shares use only the"
        f" {hot_trees['w_m'].notna().mean():.0%} with a parseable diameter_crown in"
        f" (0, {MAX_DIAMETER_M:.0f}] m."
    )

## Summary

For the release weights the test lines are skipped.

In [ ]:
gains = "; ".join(
    f"{c['from_m']:g} -> {c['to_m']:g} m: {c['diff']:.3f} [{c['ci'][0]:.3f}, {c['ci'][1]:.3f}]"
    for c in check["comparisons"]
)
summary = {
    "model": MODEL,
    "window (m)": f"{WINDOW_M:g}",
    "val mean F1 gain of each larger window over the pick, paired 95% CI": gains or "single window",
    "chosen on val": f"score > {setting['score']}, gap {setting['gap_m']} m, N {setting['min_crowns']}",
}
if MODEL != "release":
    ratio = test["diameter_ratio"]
    summary |= {
        "test tree F1, all": f1_with_ci(test["all"]["trees"]),
        "test tree F1, dense tiles": f1_with_ci(test["dense"]["trees"]),
        "test group F1 @0.4, all": f1_with_ci(test["all"]["groups_40"]),
        "test group F1 @0.4, all, slivers counted": f1_with_ci(test["all"]["groups_40_all"]),
        "test group F1 @0.4, dense tiles": f1_with_ci(test["dense"]["groups_40"]),
        "test dense tiles (units)": f"{test['dense']['tiles']} ({test['dense']['units']})",
        "box / HOT diameter, median [IQR]": f"{ratio['median']:.2f} [{ratio['q25']:.2f}, {ratio['q75']:.2f}]",
    }
summary |= {
    "HOT train area inside groups": f"{cost['group_area_share']:.1%}",
    "HOT train trees inside groups": f"{cost['trees_in_groups']:,} of {cost['trees_total']:,}",
}
if RUN_AUDIT:
    for source, row in audit.iterrows():
        summary[f"{source}: hot_rule_group_share"] = f"{row['hot_rule_group_share']:.3f}"
        summary[f"{source}: groups_over_40m"] = f"{row['groups_over_40m']:.3f}"
        summary[f"{source}: group_crowns_equiv_median"] = f"{row['group_crowns_equiv_median']:.1f}"
print(pd.Series(summary).to_string())

drive.flush_and_unmount()